In [4]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 6, Finished, Available, Finished, False)

In [5]:
t100_df = spark.read.table("bronze_t100_domestic_segment")

dim_airline = spark.read.table("dim_airline")
dim_airport = spark.read.table("dim_airport")
dim_route = spark.read.table("dim_route")
dim_date = spark.read.table("dim_date")

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 7, Finished, Available, Finished, False)

In [6]:
print("T100 Rows :", t100_df.count())

display(t100_df.limit(10))

t100_df.printSchema()

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 8, Finished, Available, Finished, False)

T100 Rows : 36186


SynapseWidget(Synapse.DataFrame, 3700ce77-4cdc-4c6f-9ab1-c39a77233064)

root
 |-- departures_scheduled: long (nullable = true)
 |-- payload: long (nullable = true)
 |-- seats: long (nullable = true)
 |-- PASSENGERS: long (nullable = true)
 |-- FREIGHT: long (nullable = true)
 |-- MAIL: long (nullable = true)
 |-- DISTANCE: long (nullable = true)
 |-- RAMP_TO_RAMP: long (nullable = true)
 |-- AIR_TIME: long (nullable = true)
 |-- UNIQUE_CARRIER: string (nullable = true)
 |-- AIRLINE_ID: long (nullable = true)
 |-- UNIQUE_CARRIER_NAME: string (nullable = true)
 |-- CARRIER: string (nullable = true)
 |-- CARRIER_NAME: string (nullable = true)
 |-- ORIGIN_AIRPORT_ID: long (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST_AIRPORT_ID: long (nullable = true)
 |-- DEST: string (nullable = true)
 |-- AIRCRAFT_TYPE: long (nullable = true)
 |-- YEAR: long (nullable = true)
 |-- QUARTER: long (nullable = true)
 |-- MONTH: long (nullable = true)



In [7]:
t100_df = (
    t100_df
    .withColumnRenamed("DEPARTURES_SCHEDULED","departures_scheduled")
    .withColumnRenamed("PAYLOAD","payload")
    .withColumnRenamed("SEATS","seats")
    .withColumnRenamed("PASSENGERS","passengers")
    .withColumnRenamed("FREIGHT","freight")
    .withColumnRenamed("MAIL","mail")
    .withColumnRenamed("DISTANCE","distance")
    .withColumnRenamed("RAMP_TO_RAMP","ramp_to_ramp")
    .withColumnRenamed("AIR_TIME","air_time")
    .withColumnRenamed("UNIQUE_CARRIER","unique_carrier")
    .withColumnRenamed("AIRLINE_ID","airline_id")
    .withColumnRenamed("UNIQUE_CARRIER_NAME","unique_carrier_name")
    .withColumnRenamed("CARRIER","carrier")
    .withColumnRenamed("CARRIER_NAME","carrier_name")
    .withColumnRenamed("ORIGIN_AIRPORT_ID","origin_airport_id")
    .withColumnRenamed("ORIGIN","origin")
    .withColumnRenamed("DEST_AIRPORT_ID","dest_airport_id")
    .withColumnRenamed("DEST","destination")
    .withColumnRenamed("AIRCRAFT_TYPE","aircraft_type")
    .withColumnRenamed("YEAR","year")
    .withColumnRenamed("QUARTER","quarter")
    .withColumnRenamed("MONTH","month")
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 9, Finished, Available, Finished, False)

In [8]:
t100_df = (
    t100_df
    .withColumn(
        "operation_date",
        to_date(
            concat_ws(
                "-",
                col("year"),
                lpad(col("month"),2,"0"),
                lit("01")
            )
        )
    )
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 10, Finished, Available, Finished, False)

In [9]:
airline_lookup = (
    dim_airline
    .select(
        "airline_key",
        col("iata_code").alias("unique_carrier")
    )
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 11, Finished, Available, Finished, False)

In [10]:
t100_df = (
    t100_df
    .join(
        airline_lookup,
        on="unique_carrier",
        how="left"
    )
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 12, Finished, Available, Finished, False)

In [11]:
print(
    "Null Airline Keys:",
    t100_df.filter(col("airline_key").isNull()).count()
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 13, Finished, Available, Finished, False)

Null Airline Keys: 2243


In [12]:
t100_df = (
    t100_df
    .filter(col("airline_key").isNotNull())
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 14, Finished, Available, Finished, False)

In [13]:
origin_lookup = (
    dim_airport
    .select(
        "airport_key",
        col("iata_code").alias("origin")
    )
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 15, Finished, Available, Finished, False)

In [14]:
t100_df = (
    t100_df
    .join(
        origin_lookup,
        on="origin",
        how="left"
    )
    .withColumnRenamed(
        "airport_key",
        "origin_airport_key"
    )
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 16, Finished, Available, Finished, False)

In [15]:
print(
    "Null Origin Airport Keys:",
    t100_df.filter(col("origin_airport_key").isNull()).count()
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 17, Finished, Available, Finished, False)

Null Origin Airport Keys: 661


In [16]:
destination_lookup = (
    dim_airport
    .select(
        "airport_key",
        col("iata_code").alias("destination")
    )
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 18, Finished, Available, Finished, False)

In [17]:
t100_df = (
    t100_df
    .join(
        destination_lookup,
        on="destination",
        how="left"
    )
    .withColumnRenamed(
        "airport_key",
        "destination_airport_key"
    )
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 19, Finished, Available, Finished, False)

In [18]:
print(
    "Null Destination Airport Keys:",
    t100_df.filter(col("destination_airport_key").isNull()).count()
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 20, Finished, Available, Finished, False)

Null Destination Airport Keys: 645


In [19]:
t100_df = (
    t100_df
    .filter(col("origin_airport_key").isNotNull())
    .filter(col("destination_airport_key").isNotNull())
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 21, Finished, Available, Finished, False)

In [20]:
print("Rows After Filtering:", t100_df.count())

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 22, Finished, Available, Finished, False)

Rows After Filtering: 36294


In [21]:
route_lookup = (
    dim_route
    .select(
        "route_key",
        col("origin_airport").alias("origin"),
        col("destination_airport").alias("destination")
    )
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 23, Finished, Available, Finished, False)

In [22]:
t100_df = (
    t100_df
    .join(
        route_lookup,
        ["origin", "destination"],
        "left"
    )
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 24, Finished, Available, Finished, False)

In [23]:
print(
    "Null Route Keys:",
    t100_df.filter(col("route_key").isNull()).count()
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 25, Finished, Available, Finished, False)

Null Route Keys: 0


In [26]:
dim_date.printSchema()

display(dim_date.limit(5))

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 28, Finished, Available, Finished, False)

root
 |-- flight_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- date_key: integer (nullable = true)



SynapseWidget(Synapse.DataFrame, 520d1af5-7f57-4fa9-b151-b91cc396e82c)

In [27]:
date_lookup = (
    dim_date
    .select(
        "date_key",
        col("flight_date").alias("operation_date")
    )
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 29, Finished, Available, Finished, False)

In [28]:
t100_df = (
    t100_df
    .join(
        date_lookup,
        on="operation_date",
        how="left"
    )
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 30, Finished, Available, Finished, False)

In [29]:
print(
    "Null Date Keys:",
    t100_df.filter(col("date_key").isNull()).count()
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 31, Finished, Available, Finished, False)

Null Date Keys: 0


In [30]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.orderBy(
    "date_key",
    "airline_key",
    "origin_airport_key",
    "destination_airport_key",
    "aircraft_type"
)

fact_airline_traffic = (
    t100_df
    .withColumn(
        "traffic_key",
        row_number().over(window_spec)
    )
    .select(
        "traffic_key",
        "date_key",
        "airline_key",
        "origin_airport_key",
        "destination_airport_key",
        "route_key",

        "aircraft_type",

        "departures_scheduled",

        "payload",
        "seats",
        "passengers",
        "freight",
        "mail",

        "distance",
        "ramp_to_ramp",
        "air_time"
    )
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 32, Finished, Available, Finished, False)

In [31]:
print("Rows :", fact_airline_traffic.count())

fact_airline_traffic.printSchema()

display(fact_airline_traffic.limit(10))

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 33, Finished, Available, Finished, False)

Rows : 36294
root
 |-- traffic_key: integer (nullable = false)
 |-- date_key: integer (nullable = true)
 |-- airline_key: integer (nullable = true)
 |-- origin_airport_key: integer (nullable = true)
 |-- destination_airport_key: integer (nullable = true)
 |-- route_key: integer (nullable = true)
 |-- aircraft_type: long (nullable = true)
 |-- departures_scheduled: long (nullable = true)
 |-- payload: long (nullable = true)
 |-- seats: long (nullable = true)
 |-- passengers: long (nullable = true)
 |-- freight: long (nullable = true)
 |-- mail: long (nullable = true)
 |-- distance: long (nullable = true)
 |-- ramp_to_ramp: long (nullable = true)
 |-- air_time: long (nullable = true)



SynapseWidget(Synapse.DataFrame, 3990bde0-be9e-4a72-9c32-2d9c16b8ab86)

In [32]:
print(
    "Duplicate Traffic Keys:",
    fact_airline_traffic.groupBy("traffic_key").count().filter("count > 1").count()
)

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 34, Finished, Available, Finished, False)

Duplicate Traffic Keys: 0


In [33]:
fact_airline_traffic.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_airline_traffic")

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 35, Finished, Available, Finished, False)

In [34]:
spark.read.table("fact_airline_traffic").count()

StatementMeta(, 8087c0fb-deea-4a84-9d9b-26680092c887, 36, Finished, Available, Finished, False)

36294